# 1.3 Preprocessing

Required preprocessing for the COVID-19 severity dataset. The split is performed before fitting any imputer, feature selector, or scaler to avoid test-set leakage.


In [21]:
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [22]:
RANDOM_STATE = 42
TARGET_COL = "Label"
DATA_PATH = "../Covid.csv"

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()


Dataset shape: (1585, 41)


,age,gender,contact with patient,Fever,Cough,Muscular pain,Shortness of breath,Inability to wake,loss of smell,loss of taste,...,Immune Deficiency (Acquired or Congenital),Pregnancy,Heart disease,Chronic kidney disease,Asthma,Other chronic lung diseases,Chronic neurological disorders,Other chronic diseases,History of hypertension,Label
0,3,0,1,1,1,1,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,-1
1,1,0,0,0,1,0,0,0,0,0.0,...,0,0,0,0,0,0,0,0,0,-1
2,5,1,0,0,0,0,1,0,0,0.0,...,0,0,1,0,0,0,0,0,0,-1
3,1,0,1,0,0,0,0,1,0,0.0,...,0,0,0,0,0,0,0,0,0,-1
4,3,1,0,1,1,1,1,0,0,0.0,...,0,0,0,0,0,1,0,0,0,-1


## 1. Missing Value Report

The table below reports the percentage of missing values for every column. Numeric ordinal fields are imputed with the median because it is robust to skew. Binary/categorical features are imputed with the mode because it preserves the most common observed category.


In [23]:
missing_report = (
    df.isnull()
      .mean()
      .mul(100)
      .round(2)
      .rename("missing_percent")
      .reset_index()
      .rename(columns={"index": "column"})
)

missing_report


,column,missing_percent
0,age,0.00
1,gender,0.00
2,contact with patient,0.00
3,Fever,0.00
4,Cough,0.00
5,Muscular pain,0.00
6,Shortness of breath,0.00
7,Inability to wake,0.00
8,loss of smell,0.00
9,loss of taste,1.70


In [24]:
feature_cols = [col for col in df.columns if col != TARGET_COL]

# In this CSV, age is encoded as ordinal bins (0-5), while PO2 is binary.
# Treat numeric non-binary fields as median-imputed numerical features; all others are mode-imputed.
numerical_features = [
    col for col in feature_cols
    if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) > 2
]
categorical_features = [col for col in feature_cols if col not in numerical_features]

# StandardScaler is reserved for true continuous variables only. The provided CSV has no such column.
continuous_features = [
    col for col in numerical_features
    if df[col].nunique(dropna=True) > 10
]

feature_type_report = pd.DataFrame({
    "feature": feature_cols,
    "unique_non_missing_values": [df[col].nunique(dropna=True) for col in feature_cols],
    "preprocessing_type": [
        "continuous_scaled" if col in continuous_features
        else "numerical_median_imputed" if col in numerical_features
        else "categorical_or_binary_mode_imputed"
        for col in feature_cols
    ],
})

print(f"Numerical median-imputed features: {numerical_features}")
print(f"Mode-imputed categorical/binary features: {len(categorical_features)}")
print(f"Continuous scaled features: {continuous_features}")
feature_type_report


Numerical median-imputed features: ['age']
Mode-imputed categorical/binary features: 39
Continuous scaled features: []


,feature,unique_non_missing_values,preprocessing_type
0,age,6,numerical_median_imputed
1,gender,2,categorical_or_binary_mode_imputed
2,contact with patient,2,categorical_or_binary_mode_imputed
3,Fever,2,categorical_or_binary_mode_imputed
4,Cough,2,categorical_or_binary_mode_imputed
5,Muscular pain,2,categorical_or_binary_mode_imputed
6,Shortness of breath,2,categorical_or_binary_mode_imputed
7,Inability to wake,2,categorical_or_binary_mode_imputed
8,loss of smell,2,categorical_or_binary_mode_imputed
9,loss of taste,2,categorical_or_binary_mode_imputed


## 2. Stratified Train/Test Split

A stratified 70/30 split is used so the minority severe class keeps approximately the same proportion in train and test. All following preprocessing steps are fit only on the training set.


In [25]:
X = df[feature_cols]
y = df[TARGET_COL]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f"Train shape: {X_train_raw.shape}, Test shape: {X_test_raw.shape}")
print("Train class distribution:")
print(y_train.value_counts().sort_index())
print("\nTest class distribution:")
print(y_test.value_counts().sort_index())


Train shape: (1109, 40), Test shape: (476, 40)
Train class distribution:
Label
-1    1039
 1      70
Name: count, dtype: int64

Test class distribution:
Label
-1    446
 1     30
Name: count, dtype: int64


## 3. Train-Fitted Imputation

The imputers are fit on the training data only, then reused for both train and test data.


In [26]:
median_imputer = SimpleImputer(strategy="median")
mode_imputer = SimpleImputer(strategy="most_frequent")

X_train_imputed = X_train_raw.copy()
X_test_imputed = X_test_raw.copy()

if numerical_features:
    X_train_imputed[numerical_features] = median_imputer.fit_transform(
        X_train_raw[numerical_features]
    )
    X_test_imputed[numerical_features] = median_imputer.transform(
        X_test_raw[numerical_features]
    )

if categorical_features:
    X_train_imputed[categorical_features] = mode_imputer.fit_transform(
        X_train_raw[categorical_features]
    )
    X_test_imputed[categorical_features] = mode_imputer.transform(
        X_test_raw[categorical_features]
    )

assert X_train_imputed.isnull().sum().sum() == 0
assert X_test_imputed.isnull().sum().sum() == 0

imputation_series = []
if numerical_features:
    imputation_series.append(
        pd.Series(median_imputer.statistics_, index=numerical_features, name="imputation_value")
    )
if categorical_features:
    imputation_series.append(
        pd.Series(mode_imputer.statistics_, index=categorical_features, name="imputation_value")
    )

imputation_values = pd.concat(imputation_series).loc[feature_cols]
imputation_summary = pd.DataFrame({
    "feature": imputation_values.index,
    "strategy": ["median" if col in numerical_features else "mode" for col in imputation_values.index],
    "imputation_value": imputation_values.values,
})

imputation_summary


,feature,strategy,imputation_value
0,age,median,3.0
1,gender,mode,1.0
2,contact with patient,mode,0.0
3,Fever,mode,0.0
4,Cough,mode,1.0
5,Muscular pain,mode,0.0
6,Shortness of breath,mode,0.0
7,Inability to wake,mode,0.0
8,loss of smell,mode,0.0
9,loss of taste,mode,0.0


## 4. Kendall Correlation Feature Selection

Kendall rank correlations are computed on the imputed training data. For every pair with `|tau| > 0.5`, one feature is removed; the retained feature is the one with the larger absolute Kendall correlation with the label.


In [27]:
def kendall_feature_selection(X_train, y_train, threshold=0.5):
    kendall_corr = X_train.corr(method="kendall").abs().fillna(0.0)
    target_corr = X_train.apply(lambda col: col.corr(y_train, method="kendall")).abs().fillna(0.0)

    removed_features = set()
    removal_records = []
    columns = list(X_train.columns)

    # Handle strongest correlated pairs first for deterministic, stable selection.
    candidate_pairs = []
    for i, feature_a in enumerate(columns):
        for feature_b in columns[i + 1:]:
            tau_abs = kendall_corr.loc[feature_a, feature_b]
            if tau_abs > threshold:
                candidate_pairs.append((tau_abs, feature_a, feature_b))

    candidate_pairs.sort(reverse=True)

    for tau_abs, feature_a, feature_b in candidate_pairs:
        if feature_a in removed_features or feature_b in removed_features:
            continue

        corr_a = target_corr.loc[feature_a]
        corr_b = target_corr.loc[feature_b]

        if corr_a >= corr_b:
            kept, removed = feature_a, feature_b
            kept_corr, removed_corr = corr_a, corr_b
        else:
            kept, removed = feature_b, feature_a
            kept_corr, removed_corr = corr_b, corr_a

        removed_features.add(removed)
        removal_records.append({
            "removed_feature": removed,
            "kept_feature": kept,
            "abs_kendall_between_features": tau_abs,
            "removed_abs_corr_with_label": removed_corr,
            "kept_abs_corr_with_label": kept_corr,
        })

    selected_features = [col for col in columns if col not in removed_features]
    removal_report = pd.DataFrame(removal_records)
    return selected_features, removal_report, kendall_corr, target_corr

selected_features, kendall_removal_report, kendall_corr_train, corr_with_label = kendall_feature_selection(
    X_train_imputed,
    y_train,
    threshold=0.5,
)

print(f"Original number of features: {len(feature_cols)}")
print(f"Remaining features after Kendall selection: {len(selected_features)}")
print(f"Removed features: {len(kendall_removal_report)}")
selected_features


Original number of features: 40
Remaining features after Kendall selection: 39
Removed features: 1


['age',
 'gender',
 'contact with patient',
 'Fever',
 'Cough',
 'Muscular pain',
 'Shortness of breath',
 'Inability to wake',
 'loss of smell',
 'loss of taste',
 'Convulsions',
 'headache',
 'dizziness',
 'paresis',
 'plexus',
 'Chest pain',
 'inflammation / skin lesions',
 'Stomachache',
 'nausea',
 'diarrhea',
 'loss of appetite',
 'smoking',
 'History of drug abuse (opium)',
 'Intubation',
 'PO2',
 'cancer',
 'Chronic liver disease',
 'Diabetes',
 'Chronic blood diseases',
 'HIV/AIDS',
 'Immune Deficiency (Acquired or Congenital)',
 'Pregnancy',
 'Heart disease ',
 'Chronic kidney disease',
 'Asthma',
 'Other chronic lung diseases',
 'Chronic neurological disorders',
 'Other chronic diseases',
 'History of hypertension']

In [28]:
kendall_removal_report


,removed_feature,kept_feature,abs_kendall_between_features,removed_abs_corr_with_label,kept_abs_corr_with_label
0,vomiting,nausea,0.766494,0.075083,0.075083


## 5. Scaling Continuous Features

Only retained true continuous features are standardized. The provided CSV encodes `age` as ordinal bins and `PO2` as binary, so this step may correctly leave the data unchanged. If a continuous feature is present and retained, the scaler is fit on the training split and then applied to the test split.


In [29]:
X_train_selected = X_train_imputed[selected_features].copy()
X_test_selected = X_test_imputed[selected_features].copy()

continuous_selected = [col for col in continuous_features if col in selected_features]
scaler = StandardScaler()

X_train_preprocessed = X_train_selected.copy()
X_test_preprocessed = X_test_selected.copy()

if continuous_selected:
    X_train_preprocessed[continuous_selected] = scaler.fit_transform(
        X_train_selected[continuous_selected]
    )
    X_test_preprocessed[continuous_selected] = scaler.transform(
        X_test_selected[continuous_selected]
    )

print(f"Scaled continuous features: {continuous_selected}")
print(f"Final train shape: {X_train_preprocessed.shape}")
print(f"Final test shape: {X_test_preprocessed.shape}")


Scaled continuous features: []
Final train shape: (1109, 39)
Final test shape: (476, 39)


## 6. Final Preprocessing Summary


In [30]:
preprocessing_summary = {
    "missing_report": missing_report,
    "feature_type_report": feature_type_report,
    "imputation_summary": imputation_summary,
    "selected_features": selected_features,
    "kendall_removal_report": kendall_removal_report,
    "continuous_scaled": continuous_selected,
    "train_shape": X_train_preprocessed.shape,
    "test_shape": X_test_preprocessed.shape,
}

print("Preprocessing completed without leakage:")
print("- Missing rates were reported on the raw data.")
print("- Train/test split used stratification with random_state=42.")
print("- Imputers, Kendall selection, and scaler were fit only on training data.")
print("- Test data was transformed using the fitted training preprocessing objects.")


Preprocessing completed without leakage:
- Missing rates were reported on the raw data.
- Train/test split used stratification with random_state=42.
- Imputers, Kendall selection, and scaler were fit only on training data.
- Test data was transformed using the fitted training preprocessing objects.
